# SVM — TF-IDF

Step 3 of the comparison. Unlike `MultinomialNB` (Step 2), a linear SVM has no assumption that
favors raw counts over TF-IDF's re-weighting — it works directly on whatever real-valued feature
space it's given, and TF-IDF's down-weighting of ubiquitous tokens is if anything a help rather
than a mismatch. So only one vectorizer is tested here, `TfidfVectorizer`, the standard pairing —
there's no equivalent "check the textbook claim" question the way there was for NB in Step 2.

Same `ngram_range=(1, 2)` as Step 2, same reasoning: stopwords (including negation words like
`không`) were kept in Step 1, and bigrams are what actually lets that pay off (`không_thích` as its
own feature, not just `không` and `thích` independently) — see `Personal Note.md` for the full
reasoning, including why full negation-scope tagging wasn't built instead.

Same `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` as Step 2 — identical folds
(same seed, same data order), so this is compared against both NB variants on the exact same
splits, not just a similar setup.

## Imports

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

## Load Preprocessed Data

In [2]:
PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUTS_DIR = os.path.join("..", "data", "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(PROCESSED_DIR, "reviews.parquet"))
X = df["clean_comment"].values
y = df["label"].values

print(df.shape)

## Cross-Validation Setup

In [3]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

## CV Loop

Same `run_cv`/`summarize` shape as `02_naive_bayes.ipynb` (each notebook stays runnable on its
own, no cross-notebook imports — the repo convention, see `README.md`), swapped to `LinearSVC`.

In [4]:
def run_cv(vectorizer_factory, model_factory, X, y, skf):
    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        vectorizer = vectorizer_factory()
        X_train = vectorizer.fit_transform(X[train_idx])
        X_val = vectorizer.transform(X[val_idx])
        y_train, y_val = y[train_idx], y[val_idx]

        model = model_factory()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_val, y_pred, average="macro", zero_division=0
        )
        fold_metrics.append({
            "fold": fold,
            "vocab_size": len(vectorizer.vocabulary_),
            "accuracy": accuracy_score(y_val, y_pred),
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1,
        })
    return fold_metrics


def summarize(fold_metrics: list[dict], method: str, vectorizer_desc: str, model: str) -> dict:
    keys = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    mean = {k: float(np.mean([f[k] for f in fold_metrics])) for k in keys}
    std = {k: float(np.std([f[k] for f in fold_metrics])) for k in keys}
    return {
        "method": method,
        "vectorizer": vectorizer_desc,
        "model": model,
        "n_splits": N_SPLITS,
        "folds": fold_metrics,
        "mean": mean,
        "std": std,
    }

## SVM + TF-IDF

### How `LinearSVC` Handles 3 Classes: One-vs-Rest (OvR)

No extra code was needed above for this to work with `label` having 3 values (0/1/2) instead of a
binary one — `LinearSVC`'s `multi_class` parameter defaults to `"ovr"` (One-vs-Rest). Calling
`.fit(X, y)` on data with 3 distinct labels makes sklearn train 3 binary hyperplanes internally,
one class versus everything else each time:

- classifier 1: Negative vs {Neutral, Positive}
- classifier 2: Neutral vs {Negative, Positive}
- classifier 3: Positive vs {Negative, Neutral}

At predict time, each of the 3 classifiers outputs one decision score (signed distance to its own
hyperplane), and the sample is assigned to whichever class's score is highest. Verified below
rather than assumed: `coef_.shape` should come out `(3, n_features)` — 3 separate hyperplanes —
and `decision_function` should return one score per class per sample.

**Not the same mechanism as `sklearn.svm.SVC`** (the kernel-based version), which always trains
One-vs-One internally regardless of settings — n×(n-1)/2 = 3 binary classifiers for 3 classes too,
coincidentally the same count exactly at 3 classes, but a different strategy that scales
differently as the number of classes grows (6 for 4 classes under OvO vs. 4 under OvR, and the gap
widens from there). `LinearSVC` defaults to OvR because it's cheaper for linear/sparse data like
TF-IDF — n classifiers instead of n×(n-1)/2, each a plain linear fit.

This is also the concrete reason `precision_recall_fscore_support(..., average="macro")` is used
for evaluation throughout Steps 2-3, not accuracy alone or micro-averaging: macro computes each
class's score independently, then averages unweighted, so the smallest class (Neutral, ~29% of the
data) can't get quietly swamped by the largest (Negative, ~36%) the way accuracy or a micro-average
would let it.

In [5]:
# Demonstration only -- fits on the full corpus just to show the OvR mechanism concretely,
# separate from the actual per-fold CV evaluation above (this fit is not used for any metric).
demo_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2).fit(X)
demo_clf = LinearSVC(random_state=42).fit(demo_vectorizer.transform(X), y)

print("classes_:", demo_clf.classes_)
print("coef_ shape (n_classes, n_features):", demo_clf.coef_.shape)
print("decision_function, one sample (one score per class):", demo_clf.decision_function(demo_vectorizer.transform(X[:1])))

In [6]:
svm_folds = run_cv(
    vectorizer_factory=lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    model_factory=lambda: LinearSVC(random_state=42),
    X=X, y=y, skf=skf,
)
svm_tfidf_results = summarize(
    svm_folds,
    method="svm_tfidf",
    vectorizer_desc="TfidfVectorizer(ngram_range=(1,2), min_df=2)",
    model="LinearSVC",
)

print(f"vocab size per fold: {[f['vocab_size'] for f in svm_folds]}")
for k, v in svm_tfidf_results["mean"].items():
    print(f"{k}: {v:.4f} ± {svm_tfidf_results['std'][k]:.4f}")

## Saving Metrics

In [7]:
out_path = os.path.join(OUTPUTS_DIR, "svm_tfidf_metrics.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(svm_tfidf_results, f, ensure_ascii=False, indent=2)

print(f"Saved -> {out_path}")